# Task 4 — `core/gates.py`

Walkthrough of the new gate library. Focus on physics you should verify by hand:
rotation matrices, angle convention, CNOT convention flip, qutrit gates, and utilities.

In [ ]:
import numpy as np
from qaravan.core.base import Gate, MatrixGate, ParametricGate
from qaravan.core.gates import (
    I, X, Y, Z, H, S, Sdg, T, Tdg, SX,
    RX, RY, RZ,
    CNOT, CZ, SWAP, iSWAP,
    RXX, RYY, RZZ,
    X01, SX01, X12, SX12, H01, SDG01, SWAP3, CNOT3,
    random_unitary, is_unitary, pauli_string_to_gates,
)

## 1. Gate class hierarchy

`MatrixGate` stores a fixed matrix. `ParametricGate` computes it from params.

In [ ]:
print(isinstance(H(0), Gate))       # True
print(isinstance(H(0), MatrixGate)) # True
print(isinstance(RX(0, 0.5), ParametricGate))  # True

# Ad-hoc gate
my_u = MatrixGate("U", [0, 1], random_unitary(2))
print("ad-hoc gate unitary:", is_unitary(my_u.matrix))

## 2. Rotation gate convention: `exp(-i θ P)`, full angle

At `θ = π/2`: `RX` ≈ `-iX`, `RZ` ≈ `-iZ`. At `θ = 0`: identity.

In [ ]:
print("RX(π/2) matrix:\n", np.round(RX(0, np.pi/2).matrix, 6))
print("\n-iX:\n", -1j * X(0).matrix)
print("\nMatch:", np.allclose(RX(0, np.pi/2).matrix, -1j * X(0).matrix))

In [ ]:
print("RZ(π/2) matrix:\n", np.round(RZ(0, np.pi/2).matrix, 6))
print("\n-iZ:\n", -1j * Z(0).matrix)
print("\nMatch:", np.allclose(RZ(0, np.pi/2).matrix, -1j * Z(0).matrix))

In [ ]:
# RZZ diagonal: [e^{-iθ}, e^{iθ}, e^{iθ}, e^{-iθ}]
theta = 0.7
diag = np.diag(RZZ([0,1], theta).matrix)
print("RZZ diagonal:", np.round(diag, 6))
print("Expected:", np.round([np.exp(-1j*theta), np.exp(1j*theta),
                              np.exp(1j*theta), np.exp(-1j*theta)], 6))

## 3. ParametricGate: name, str, dagger

In [ ]:
g = RX(0, np.pi/4)
print("name:", g.name)   # 'RX' — bare name, no params
print("str:", str(g))    # 'RX([0], 0.7854)' — shows name + indices + params
print("params:", g.params)

gd = g.dagger()
print("dagger type:", type(gd).__name__)  # RX, not MatrixGate
print("dagger params:", gd.params)        # (-π/4,)
print("g @ gd = I:", np.allclose(g.matrix @ gd.matrix, np.eye(2)))

## 4. CNOT convention: (control, target)

v0.1 used `(target, control)`. v0.2 uses `(control, target)`. Verify that
control=0, target=1 causes `|10⟩ → |11⟩` in big-endian qubit ordering.

In [ ]:
cnot = CNOT([0, 1])
print("CNOT matrix:\n", cnot.matrix.real.astype(int))

# |10⟩ = qubit0=1, qubit1=0 = index 2 in big-endian
psi_10 = np.array([0, 0, 1, 0], dtype=complex)
psi_out = cnot.matrix @ psi_10
print("\nCNOT|10⟩ =", psi_out)  # should be |11⟩ = index 3

# |00⟩ unchanged
psi_00 = np.array([1, 0, 0, 0], dtype=complex)
print("CNOT|00⟩ =", cnot.matrix @ psi_00)

## 5. Single-qubit gate inverses

In [ ]:
for name, g in [("H", H(0)), ("S†S", Sdg(0).matrix @ S(0).matrix),
                 ("T†T", Tdg(0).matrix @ T(0).matrix)]:
    if isinstance(g, Gate):
        print(f"{name} self-inverse: {np.allclose(g.dagger().matrix, g.matrix)}")
    else:
        print(f"{name} = I: {np.allclose(g, np.eye(2))}")

## 6. Qutrit gates

All should be unitary with `local_dim=3`.

In [ ]:
qutrit_gates = [X01(0), SX01(0), X12(0), SX12(0), H01(0), SDG01(0),
                SWAP3([0,1]), CNOT3([0,1])]
for g in qutrit_gates:
    print(f"{g.name:8s}  local_dim={g.local_dim}  unitary={is_unitary(g.matrix)}")

## 7. Utilities

In [ ]:
# random_unitary: seed-reproducible, Haar-random
u1 = random_unitary(2, seed=42)
u2 = random_unitary(2, seed=42)
print("same seed → same unitary:", np.allclose(u1, u2))
print("is unitary:", is_unitary(u1))

# pauli_string_to_gates
gates = pauli_string_to_gates("XZI")
print("\npauli_string_to_gates('XZI'):")
for g in gates:
    print(" ", g, "matrix:", g.matrix)